# Pipeline Walkthrough — Sample End-to-End

Walks through the pipeline's data sources on small samples so you can see what each API actually returns and how we combine them.  No DB writes, no S3, no AWS.

The last section is the interesting one for the ORCID-coverage question: it tries a plain **name + PSU affiliation** search against OpenAlex on a few of the 235 currently-unmapped researchers, to show whether the simpler path works before we build a fuzzy matcher.

## Setup

Pulls API keys from AWS Secrets Manager via the instance role.  If you're running outside the EC2 dev box, set `PSU_RESEARCH_API_KEY` and `OPENALEX_EMAIL` in your environment instead.

In [ ]:
import json, os, time
from pathlib import Path
import requests

try:
    import boto3
    sm = boto3.client('secretsmanager', region_name='us-east-1')
    RMD_KEY = os.environ.get('PSU_RESEARCH_API_KEY') or sm.get_secret_value(SecretId='overton/api-keys/rmd')['SecretString']
except Exception:
    RMD_KEY = os.environ['PSU_RESEARCH_API_KEY']

OPENALEX_EMAIL = os.environ.get('OPENALEX_EMAIL', 'overton-pipeline@psu.edu')
PSU_ROR = 'https://ror.org/04p491231'

RMD_BASE = 'https://metadata.libraries.psu.edu/v1'
OA_BASE  = 'https://api.openalex.org'

def rmd_get(path, params=None, auth=True):
    h = {'Accept': 'application/json'}
    if auth:
        h['X-API-Key'] = RMD_KEY
    r = requests.get(f'{RMD_BASE}{path}', params=params, headers=h, timeout=30)
    r.raise_for_status()
    time.sleep(0.3)
    return r.json()

def oa_get(path, params=None):
    h = {'User-Agent': f'mailto:{OPENALEX_EMAIL}'}
    r = requests.get(f'{OA_BASE}{path}', params=params, headers=h, timeout=30)
    r.raise_for_status()
    time.sleep(0.1)
    return r.json()

def pretty(o, limit=1500):
    s = json.dumps(o, indent=2, default=str)
    print(s if len(s) < limit else s[:limit] + f'\n... ({len(s):,} chars total)')

print('keys loaded')

## Step 1 — RMD: list the 17 HHD organizations
The pipeline starts here: every researcher we track is someone who appears as a contributor on a publication in one of these orgs.

In [ ]:
orgs = rmd_get('/organizations')['data']
print(f'{len(orgs)} organizations')
for o in orgs[:5]:
    a = o['attributes']
    print(f'  {o["id"]:>4}  {a.get("name")}')

## Step 2 — RMD: one publication, and what fields the contributors have
This is the shape the pipeline scans 16k+ of.  Pay attention to the `contributors` array — each contributor may have `first_name`, `last_name`, `psu_user_id`, and sometimes `orcid` directly on the record.

In [ ]:
# Pick one org, grab a handful of publications
sample_org = orgs[0]['id']
pubs = rmd_get(f'/organizations/{sample_org}/publications', params={'limit': 3})['data']

pub = pubs[0]
print('Publication attributes (top-level keys):')
print(' ', list(pub['attributes'].keys()))
print()
print('First contributor entry:')
pretty(pub['attributes']['contributors'][0])

In [ ]:
# How often do contributors actually have an ORCID on the publication record itself?
batch = rmd_get(f'/organizations/{sample_org}/publications', params={'limit': 100})['data']
total_contribs = with_orcid = with_uid = 0
for p in batch:
    for c in p['attributes'].get('contributors', []):
        total_contribs += 1
        if c.get('orcid'): with_orcid += 1
        if c.get('psu_user_id'): with_uid += 1
print(f'{total_contribs} contributor rows across {len(batch)} publications')
print(f'  {with_orcid} have orcid directly on the contributor record ({with_orcid/total_contribs:.0%})')
print(f'  {with_uid} have psu_user_id ({with_uid/total_contribs:.0%})')

## Step 3 — RMD: a user profile
For each `psu_user_id` we discover, the pipeline calls `/users/{uid}/profile` to get title/org/email/etc.  The `orcid_identifier` field is what the pipeline's current "profile fallback" strategy reads.

In [ ]:
# Pick a psu_user_id from the batch above
some_uid = next(c['psu_user_id'] for p in batch for c in p['attributes'].get('contributors', []) if c.get('psu_user_id'))
print(f'Looking up user {some_uid}')
profile = rmd_get(f'/users/{some_uid}/profile', auth=False)
pretty(profile['data']['attributes'])

## Step 4 — OpenAlex: the happy path (lookup by ORCID)
When RMD gives us an ORCID, OpenAlex gives us rich academic metadata in one call.  This is the 750 we already have.

In [ ]:
# Take any ORCID we already have mapped
with open('../data/pipeline/orcid_webaccess_map.json') as f:
    orcid_map = json.load(f)

sample_orcid = next(iter(orcid_map))
print(f'Looking up ORCID {sample_orcid}')
author = oa_get(f'/authors/orcid:{sample_orcid}')
print('Selected fields:')
pretty({
    'id': author.get('id'),
    'display_name': author.get('display_name'),
    'orcid': author.get('orcid'),
    'works_count': author.get('works_count'),
    'cited_by_count': author.get('cited_by_count'),
    'h_index': author.get('summary_stats', {}).get('h_index'),
    'last_known_institutions': [i.get('display_name') for i in author.get('last_known_institutions', [])],
    'topics_top3': [t.get('display_name') for t in author.get('topics', [])[:3]],
})

## Step 5 — OpenAlex: the PSU author universe
The pipeline currently enumerates all PSU authors *with ORCIDs* and name-matches against RMD users.  Let's see the raw shape of that list.

In [ ]:
# Just fetch first page to see the shape - full enumeration is in the pipeline's --rebuild-map
resp = oa_get('/authors', params={
    'filter': f'affiliations.institution.ror:{PSU_ROR},has_orcid:true',
    'per_page': 5,
    'select': 'id,orcid,display_name,works_count,affiliations',
})
print(f'Total PSU authors with ORCIDs on OpenAlex: {resp["meta"]["count"]:,}')
print(f'\nFirst 5:')
for a in resp['results']:
    print(f'  {a["orcid"]}  {a["display_name"]}  ({a["works_count"]} works)')

## Step 6 — The interesting cell: name search for one of our 235 unmapped researchers

The pipeline's current three strategies (exact name match, DOI cross-ref, RMD profile `orcid_identifier`) all failed for 235 researchers.  None of them have an ORCID in RMD.

But OpenAlex has its own `search` parameter that handles name variations, middle initials, punctuation, and ranks results by relevance. Let's see what it returns for a few of the unmapped names — this is what the 'simpler path' would look like before we do any fuzzy scoring ourselves.

In [ ]:
# Load the coverage report
with open('../data/pipeline/orcid_coverage_report.json') as f:
    report = json.load(f)

unmapped = report['unmapped_no_profile_orcid']
print(f'{len(unmapped)} researchers with no ORCID anywhere in RMD')
print(f'Sample: {[u["name"] for u in unmapped[:8]]}')

In [ ]:
# For each of a few unmapped researchers, do the simplest possible OpenAlex lookup
# Filter: at PSU, has an ORCID. Search: their RMD name.
# This is what a 'just-try-it' approach would look like.

def search_psu_author(name, require_orcid=True):
    filter_parts = [f'affiliations.institution.ror:{PSU_ROR}']
    if require_orcid:
        filter_parts.append('has_orcid:true')
    resp = oa_get('/authors', params={
        'filter': ','.join(filter_parts),
        'search': name,
        'per_page': 5,
        'select': 'id,orcid,display_name,display_name_alternatives,works_count,last_known_institutions',
    })
    return resp['results']

# Try 8 unmapped names — adjust the slice to see more
for u in unmapped[:8]:
    name = u['name']
    hits = search_psu_author(name)
    print(f'\n{name!r}  (uid={u["uid"]})')
    if not hits:
        # Fall back to no-ORCID filter to see if they exist on OpenAlex at all
        any_hits = search_psu_author(name, require_orcid=False)
        if any_hits:
            print(f'  (no ORCID hits, but {len(any_hits)} PSU author(s) match without ORCID filter — first: {any_hits[0]["display_name"]})')
        else:
            print('  no match at all')
    else:
        for h in hits:
            inst = h.get('last_known_institutions') or []
            inst_name = inst[0]['display_name'] if inst else '-'
            print(f'  -> {h["orcid"]}  {h["display_name"]:<35}  works={h["works_count"]:<4}  inst={inst_name}')

### What to look for in the cell above

Three outcomes per name:

1. **One clear PSU author with ORCID ranks first** → these are easy wins. We could auto-accept when the top hit has high name overlap with the RMD name and the works count is non-trivial.
2. **Multiple candidates** → ambiguous; would need disambiguation (e.g., overlap with the researcher's RMD DOIs).
3. **No ORCID hits at all** → the researcher either isn't on OpenAlex or has no ORCID registered anywhere. Unreachable without a better data source.

The ratio between these three buckets tells us whether a simple name-search approach is enough or whether we need the heavier fuzzy-scoring-plus-DOI-cross-reference strategy.

## Step 7 — OpenAlex: lookup by DOI (for context on the DOI-matching path)
Just to see why DOI cross-reference is heavier: for each unmapped user we'd need to resolve several of their RMD DOIs, then examine each work's authorships.  One call per DOI.

In [ ]:
# Grab a DOI from one of the publications we fetched
sample_doi = next(p['attributes']['doi'] for p in batch if p['attributes'].get('doi'))
print(f'DOI: {sample_doi}')
work = oa_get(f'/works/doi:{sample_doi}')
print(f'Title: {work.get("title")}')
print(f'\nAuthorships:')
for a in work.get('authorships', [])[:10]:
    au = a.get('author', {})
    print(f'  {au.get("orcid") or "(no orcid)":<45}  {au.get("display_name")}')

## Step 8 — Combine: a single unified researcher record
Pull from RMD + OpenAlex for one researcher and assemble the shape the pipeline stores in the `researchers.data` JSONB column.

In [ ]:
# Take a known-mapped pair (ORCID + webaccess_id) from the map
sample_orcid, sample_uid = next(iter(orcid_map.items()))

oa = oa_get(f'/authors/orcid:{sample_orcid}')
profile = rmd_get(f'/users/{sample_uid}/profile', auth=False)['data']['attributes']
grants = rmd_get(f'/users/{sample_uid}/grants')['data']

unified = {
    'orcid': sample_orcid,
    'openalex_id': oa.get('id'),
    'display_name': oa.get('display_name') or profile.get('title'),
    'openalex': {
        'works_count': oa.get('works_count'),
        'cited_by_count': oa.get('cited_by_count'),
        'h_index': oa.get('summary_stats', {}).get('h_index'),
        'topics': [t.get('display_name') for t in oa.get('topics', [])[:3]],
    },
    'rmd': {
        'webaccess_id': sample_uid,
        'title': profile.get('title'),
        'organization_name': profile.get('organization_name'),
        'email': profile.get('email'),
        'grants_count': len(grants),
        'grants_total_dollars': sum((g['attributes'].get('amount_in_dollars') or 0) for g in grants),
    },
    # Stage 3 (Overton) would add a `overton` block of policy-document citations
}
pretty(unified)

## Summary

You've now seen every input the pipeline currently combines.  For the 235-unmapped problem the notebook makes the tradeoff concrete:

- **Step 6 (name search)** is one API call per unmapped researcher.  Fast, simple, cheap.  Quality depends on how often a top-ranked PSU author with ORCID is actually the right person.
- **Step 7 (DOI cross-ref)** is several API calls per researcher, plus authorship parsing and name matching on each side.  More confident but more moving parts.

Run Step 6 against the first 20-50 unmapped names to get a feel for the hit rate before deciding which approach to build out.